In [1]:

import os
import ast
import argparse
import json
from dataclasses import dataclass
from typing import List, Dict, Any
from pandas.api.types import CategoricalDtype
import torch
from torch.utils.data import Dataset

import pandas as pd
from PIL import Image

from transformers import (
    AutoProcessor,
    AutoModelForVision2Seq,
    TrainingArguments,
    Trainer,
    set_seed,
    LogitsProcessor, 
    LogitsProcessorList
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)
import bitsandbytes as bnb  # noqa: F401 (needed for 4-bit/8-bit quantization)

In [2]:
import torch, os
from transformers import AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig
from peft import PeftModel

In [3]:
from huggingface_hub import notebook_login


In [3]:
notebook_login()

In [4]:
LETTER_VOCAB = ["A", "B", "C", "D", "E"]

VISION_TOKENS = ("<|vision_start|>", "<|vision_end|>", "<|image_pad|>", "<|video_pad|>")

def allowed_letter_token_ids(tok):

    ids = set()
    for L in LETTER_VOCAB:
        for pref in ["", " ", "\n"]:
            pieces = tok(pref + L, add_special_tokens=False).input_ids
            if len(pieces) == 1:
                ids.add(pieces[0])
    if not ids:
        # fallback: just take the first id from encoding of the letter
        for L in LETTER_VOCAB:
            pieces = tok(L, add_special_tokens=False).input_ids
            ids.add(pieces[0])
    return sorted(ids)

# --- prompt building ----------------------------------------------------------

# def build_prompt(options: List[str], legend: bool) -> str:
#     text = (
#         "I am showing you five apartment floorplans, labeled A through E.\n"
#         "One of these plans has a different underlying floorplan pattern, while the other four share the same pattern.\n\n"
#         "The thick black outline of each of the floorplans indicates the boundary of that floorplan. "
#         "The red bar drawn on the black outline of each of the floorplans marks the main entrance of that floorplan.\n"
#     )
#     if legend:
#         text += "There is a color legend indicating the color-coding of room types below all the floorplans.\n"
#     text += (
#         "Examine each floorplan only within its thick black outer boundary, focusing on spatial layout, room types, and relative sizes.\n"
#         "Question: Which floorplan (A, B, C, D, or E) has a different underlying pattern?\n\n"
#         "Answer with a single letter only (A/B/C/D/E)."
#     )
#     return text
def build_prompt(options: list[str], legend: bool) -> str:
    COLOR_MAPPING = {
        (0xFF, 0xD7, 0x00): 'common room',
        (0xFF, 0xA5, 0x00): 'master room',
        (0xEE, 0xE8, 0xAA): 'living room',
        (0x6B, 0x8E, 0x23): 'balcony',
        (0xAD, 0xD8, 0xE6): 'bathroom',
        (0xF0, 0x80, 0x80): 'kitchen',
        (0xDD, 0xA0, 0xDD): 'storage',
        (0xDA, 0x70, 0xD6): 'dining',
    }

    text = (
        "I am showing you five apartment floorplans, labeled A through E.\n"
        "One of these plans has a different underlying floorplan pattern, while the other four share the same pattern.\n\n"
        "The thick black outline of each of the floorplans indicates the boundary of that floorplan. "
        "The red bar drawn on the black outline of each of the floorplans marks the main entrance of that floorplan.\n"
    )

    if legend:
        text += "Below is a color mapping indicating the room type for each fill color:\n"
        for rgb, room in COLOR_MAPPING.items():
            hex_code = f"#{rgb[0]:02X}{rgb[1]:02X}{rgb[2]:02X}"
            text += f"- {room}: {hex_code} (RGB{rgb})\n"
        text += "\n"

    text += (
        "Examine each floorplan **only within its thick black outer boundary**, and **use the main entrance (the red bar) "
        "as the point of entry to reorient yourself when reasoning about the spatial layout**, "
        "focusing on spatial layout, room types, and relative sizes.\n"
        "Question: Which floorplan (A, B, C, D, or E) has a different underlying pattern, and why?\n\n"
        "Please structure your response exactly as follows:\n"
        "1. **Different floorplan:** <A/B/C/D/E>\n"
        "2. **Why:** <brief reasoning>\n\n"
    )

    return text

# --- dataset ------------------------------------------------------------------

class OddOneOutDataset(Dataset):
    def __init__(self, df: pd.DataFrame, image_root: str, processor: AutoProcessor, legend: bool = False):
        self.df = df.reset_index(drop=True)
        self.image_root = image_root
        self.processor = processor
        self.legend = legend

    def __len__(self):
        return len(self.df)

    def _messages(self, image: Image.Image, prompt_text: str, answer_text: str):
        # Qwen2.5-VL chat format
        user_part = [
            {"type": "text", "text": prompt_text},
            {"type": "image", "image": image},
        ]
        assistant_part = [{"type": "text", "text": answer_text}]
        return [
            {"role": "user", "content": user_part},
            {"role": "assistant", "content": assistant_part},
        ]

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.df.iloc[idx]
        img_name = f"example_{row['orig_index']}_2.png"
        img_path = os.path.join(self.image_root, img_name)
        if not os.path.exists(img_path):
            alt_path = os.path.join(self.image_root, "datasets", "easy", img_name)
            if os.path.exists(alt_path):
                img_path = alt_path

        image = Image.open(img_path).convert("RGB")

        prompt_text = build_prompt(LETTER_VOCAB, legend=self.legend)
        answer_text = str(row["outlier_id"]).strip()

        messages_full = self._messages(image, prompt_text, answer_text)
        messages_prompt_only = [{"role": "user", "content": messages_full[0]["content"]}]

        # IMPORTANT:
        # - full text includes assistant content (the single letter)
        # - prompt-only adds the assistant preamble so generation starts at the right place
        text_full = self.processor.apply_chat_template(messages_full, tokenize=False)
        text_prompt_only = self.processor.apply_chat_template(
            messages_prompt_only, tokenize=False, add_generation_prompt=True
        )

        return {
            "image": image,
            "text_full": text_full,
            "text_prompt": text_prompt_only,
            "label_letter": answer_text,
            "idx": int(row["orig_index"]),
        }

# --- data collator (Option A) -------------------------------------------------

from dataclasses import dataclass

@dataclass
class VLDataCollator:
    processor: AutoProcessor  # Qwen2.5-VL processor

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        images = [f["image"] for f in features]
        text_full_list = [f["text_full"] for f in features]
        text_prompt_list = [f["text_prompt"] for f in features]

        # Source of truth for shapes/tokens: processor builds input_ids + vision tensors
        batch = self.processor(
            text=text_full_list,
            images=images,
            padding=True,
            return_tensors="pt",
        )

        tok = self.processor.tokenizer
        input_ids = batch["input_ids"]
        labels = input_ids.clone()

        # 1) mask padding
        pad_id = tok.pad_token_id
        if pad_id is not None:
            labels[labels == pad_id] = -100

        # 2) mask vision special tokens
        for t in VISION_TOKENS:
            tid = tok.convert_tokens_to_ids(t)
            if tid is not None and tid != -1:
                labels[labels == tid] = -100

        # 3) mask everything before the assistant reply
        #    (tprompt includes assistant prefix via add_generation_prompt=True)
        for i, tprompt in enumerate(text_prompt_list):
            n = len(tok(tprompt, add_special_tokens=False).input_ids)
            labels[i, :n] = -100

        batch["labels"] = labels
        return batch

# --- model / processor loader -------------------------------------------------

def get_model_and_processor(model_name: str, bnb_4bit: bool = True):
    processor = AutoProcessor.from_pretrained(model_name, trust_remote_code=True)

    if bnb_4bit:
        model = AutoModelForVision2Seq.from_pretrained(
            model_name,
            trust_remote_code=True,
            device_map="auto",
            torch_dtype=torch.bfloat16,
            quantization_config=dict(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16,
            ),
        )
        model = prepare_model_for_kbit_training(model)
    else:
        model = AutoModelForVision2Seq.from_pretrained(
            model_name,
            trust_remote_code=True,
            device_map="auto",
            torch_dtype=torch.bfloat16,
        )

    # LoRA on language submodules (optionally add projector via modules_to_save=["mm_projector"])
    lora_cfg = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        modules_to_save=None,
    )
    model = get_peft_model(model, lora_cfg)
    return model, processor

# --- evaluation with constrained decoding ------------------------------------

from transformers import LogitsProcessor, LogitsProcessorList

class AllowOnlyTokens(LogitsProcessor):
    def __init__(self, allowed_ids: List[int]):
        self.allowed_ids = None
        # to tensor lazily since device/shape comes at call-time
        self._ids_list = allowed_ids

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor) -> torch.FloatTensor:
        if self.allowed_ids is None or self.allowed_ids.device != scores.device:
            self.allowed_ids = torch.tensor(self._ids_list, device=scores.device, dtype=torch.long)
        mask = torch.full_like(scores, float("-inf"))
        mask.scatter_(1, self.allowed_ids.view(1, -1), scores.index_select(1, self.allowed_ids))
        return mask

def run_eval(model, processor, df_eval: pd.DataFrame, image_root: str, legend: bool, max_samples: int = None):
    model.eval()
    ds = OddOneOutDataset(df_eval if max_samples is None else df_eval.iloc[:max_samples], image_root, processor, legend)
    correct = 0
    total = 0
    preds, gts, idxs = [], [], []


    allowed_ids = allowed_letter_token_ids(processor.tokenizer)
    lp = LogitsProcessorList([AllowOnlyTokens(allowed_ids)])

    for i in range(len(ds)):
        ex = ds[i]
        inputs = processor(
            text=[ex["text_prompt"]], 
            images=[ex["image"]],
            padding=True,
            return_tensors="pt",
        ).to(model.device)

        with torch.inference_mode():
            gen = model.generate(
                **inputs,
                max_new_tokens=1,       
                do_sample=False,
                logits_processor=lp,        # force A–E
            )

        new_tokens = gen[0, inputs["input_ids"].shape[1]:]
        out = processor.tokenizer.decode(new_tokens, skip_special_tokens=True)

        pred_letter = None
        for ch in out:
            if ch in LETTER_VOCAB:
                pred_letter = ch
                break

        gt = ex["label_letter"]
        if pred_letter == gt:
            correct += 1
        total += 1
        preds.append(pred_letter if pred_letter is not None else "")
        gts.append(gt)
        idxs.append(ex["idx"])

    acc = correct / max(1, total)
    return {"accuracy": acc, "preds": preds, "gts": gts, "idxs": idxs}

In [5]:
import argparse
import shlex
import textwrap

def build_parser():
    p = argparse.ArgumentParser()
    p.add_argument("--csv", required=True)
    p.add_argument("--image_root", required=True)
    p.add_argument("--model_name", required=True)
    p.add_argument("--output_dir", required=True)
    p.add_argument("--legend_in_prompt", action="store_true")
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--num_train_epochs", type=int, default=3)
    p.add_argument("--per_device_train_batch_size", type=int, default=8)
    p.add_argument("--per_device_eval_batch_size", type=int, default=8)
    p.add_argument("--gradient_accumulation_steps", type=int, default=1)
    p.add_argument("--learning_rate", type=float, default=5e-5)
    p.add_argument("--warmup_ratio", type=float, default=0.0)
    p.add_argument("--weight_decay", type=float, default=0.0)
    p.add_argument("--logging_steps", type=int, default=50)
    p.add_argument("--save_strategy", choices=["no","steps","epoch"], default="steps")
    p.add_argument("--evaluation_strategy", choices=["no","steps","epoch"], default="no")
    p.add_argument("--bf16", action="store_true")
    return p

arg_str = textwrap.dedent("""
--csv difficult_dataset_mid_train_test.csv
--image_root ./datasets/difficult
--model_name Qwen/Qwen2.5-VL-7B-Instruct
--output_dir ./qwen2p5_vl_odd1out
--legend_in_prompt
--seed 7
--num_train_epochs 2
--per_device_train_batch_size 1
--per_device_eval_batch_size 1
--gradient_accumulation_steps 8
--learning_rate 1e-4
--warmup_ratio 0.03
--weight_decay 0.0
--logging_steps 10
--save_strategy epoch
--evaluation_strategy epoch
--bf16

""").strip()

parser = build_parser()
args = parser.parse_args(shlex.split(arg_str))

In [7]:

set_seed(args.seed)
max_train_samples = 5000

df = pd.read_csv(args.csv)
# Basic sanitation
assert {"split", "orig_index", "outlier_id"}.issubset(df.columns), \
    "CSV must contain 'split', 'orig_index' and 'outlier_id' columns."

# # Keep only A-E rows
# df = df[df["outlier_id"].isin(LETTER_VOCAB)].copy()
LABELS = ['A','B','C','D','E']
cat = CategoricalDtype(categories=LABELS, ordered=True)

def outlier_to_letter(row):
    outlier = str(row['outlier_id'])
    opts = row['options']

    # Coerce "['...']" strings to lists if needed
    if isinstance(opts, str):
        try:
            opts = ast.literal_eval(opts)
        except Exception:
            return pd.NA
    if not isinstance(opts, (list, tuple)):
        return pd.NA

    # Normalize and only consider the first 5 (A–E)
    opts = [str(x) for x in opts][:5]
    try:
        idx = opts.index(outlier)  # 0..4 only
        return LABELS[idx]
    except ValueError:
        return pd.NA

df['outlier_id'] = df.apply(outlier_to_letter, axis=1).astype(cat)

train_df = df[df["split"].str.lower().isin(["train", "training"])]
eval_df = df[df["split"].str.lower().isin(["test", "eval", "validation", "val"])]

if max_train_samples:
    train_df = train_df.sample(n=min(max_train_samples, len(train_df)), random_state=args.seed)

train_df['orig_index'] += 1
eval_df['orig_index'] += 1

/tmp/ipykernel_76761/2127253667.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eval_df['orig_index'] += 1


In [22]:
eval_df

,base_cluster,other_cluster,group,outlier_id,options,orig_index,_pair,split
24,"[17, 21]","[7, 3]","['62454', '69704', '62549', '9795', '64781']",E,"['62454', '9795', '69704', '62549', '64781']",3957,7_17,test
26,"[7, 1]","[17, 21]","['3536', '53652', '12421', '10825', '40487']",A,"['40487', '3536', '53652', '12421', '10825']",645,7_17,test
28,"[17, 18]","[7, 22]","['66935', '6955', '21543', '21408', '23132']",A,"['23132', '21408', '6955', '66935', '21543']",3615,7_17,test
32,"[17, 18]","[7, 1]","['21408', '70105', '66316', '21543', '12376']",D,"['70105', '21408', '66316', '12376', '21543']",4271,7_17,test
33,"[17, 18]","[7, 20]","['21543', '21408', '66316', '46660', '2425']",A,"['2425', '66316', '21408', '46660', '21543']",4427,7_17,test
...,...,...,...,...,...,...,...,...
4970,"[17, 21]","[7, 1]","['9795', '62549', '78147', '69704', '12376']",C,"['78147', '62549', '12376', '69704', '9795']",3500,7_17,test
4982,"[17, 18]","[7, 1]","['51240', '21543', '21408', '66935', '12421']",D,"['66935', '21543', '21408', '12421', '51240']",4295,7_17,test
4983,"[17, 18]","[7, 3]","['51240', '66316', '31693', '46660', '64781']",B,"['31693', '64781', '66316', '51240', '46660']",866,7_17,test
4985,"[7, 1]","[17, 5]","['53652', '12421', '8109', '10825', '72055']",C,"['53652', '8109', '72055', '12421', '10825']",3045,7_17,test


In [5]:

# Load model & processor with QLoRA
model, processor = get_model_and_processor(args.model_name, bnb_4bit=True)

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.
/home/airlay88/planscape/venv/lib/python3.10/site-packages/transformers/models/auto/modeling_auto.py:2199: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [6]:


# Datasets
train_dataset = OddOneOutDataset(train_df, args.image_root, processor, legend=args.legend_in_prompt)
eval_dataset = OddOneOutDataset(eval_df, args.image_root, processor, legend=args.legend_in_prompt)

collator = VLDataCollator(processor)

training_args = TrainingArguments(
    output_dir=args.output_dir,
    num_train_epochs=args.num_train_epochs,
    per_device_train_batch_size=args.per_device_train_batch_size,
    per_device_eval_batch_size=args.per_device_eval_batch_size,
    gradient_accumulation_steps=args.gradient_accumulation_steps,
    learning_rate=args.learning_rate,
    warmup_ratio=args.warmup_ratio,
    weight_decay=args.weight_decay,
    logging_steps=args.logging_steps,
    save_strategy=args.save_strategy,
    eval_strategy=args.evaluation_strategy,
    bf16=args.bf16,
    dataloader_pin_memory=False,
    remove_unused_columns=False,  
    report_to="none",
)

model.config.use_cache = False

NameError: name 'train_df' is not defined

In [15]:

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collator,
)


In [16]:
# Train
train_result = trainer.train()
trainer.save_model(args.output_dir)
if trainer.is_world_process_zero():
    metrics = train_result.metrics
    trainer.log_metrics("train", metrics)
    trainer.save_metrics("train", metrics)
    trainer.save_state()

# Quick evaluation pass with greedy decoding
if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, args.image_root, args.legend_in_prompt)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(args.output_dir, "eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(args.output_dir, "eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", args.output_dir)

/home/airlay88/planscape/venv/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/airlay88/planscape/venv/lib/python3.10/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,0.059300,0.322566
2,0.000000,0.421540


/home/airlay88/planscape/venv/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/airlay88/planscape/venv/lib/python3.10/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


***** train metrics *****
  epoch                    =         2.0
  total_flos               = 305171535GF
  train_loss               =      0.7799
  train_runtime            =  1:55:14.90
  train_samples_per_second =       1.141
  train_steps_per_second   =       0.143


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.4957 on 1055 examples
Done.
Adapter weights are saved under: ./qwen2p5_vl_odd1out


In [7]:
df = pd.read_csv("./difficult_dataset_mid_train_test.csv")
# Basic sanitation
assert {"split", "orig_index", "outlier_id"}.issubset(df.columns), \
    "CSV must contain 'split', 'orig_index' and 'outlier_id' columns."

# # Keep only A-E rows
# df = df[df["outlier_id"].isin(LETTER_VOCAB)].copy()
LABELS = ['A','B','C','D','E']
cat = CategoricalDtype(categories=LABELS, ordered=True)

def outlier_to_letter(row):
    outlier = str(row['outlier_id'])
    opts = row['options']

    # Coerce "['...']" strings to lists if needed
    if isinstance(opts, str):
        try:
            opts = ast.literal_eval(opts)
        except Exception:
            return pd.NA
    if not isinstance(opts, (list, tuple)):
        return pd.NA

    # Normalize and only consider the first 5 (A–E)
    opts = [str(x) for x in opts][:5]
    try:
        idx = opts.index(outlier)  # 0..4 only
        return LABELS[idx]
    except ValueError:
        return pd.NA

df['outlier_id'] = df.apply(outlier_to_letter, axis=1).astype(cat)

train_df = df[df["split"].str.lower().isin(["train", "training"])]
eval_df = df[df["split"].str.lower().isin(["test", "eval", "validation", "val"])]

if max_train_samples:
    train_df = train_df.sample(n=min(max_train_samples, len(train_df)), random_state=args.seed)

train_df['orig_index'] += 1
eval_df['orig_index'] += 1

NameError: name 'max_train_samples' is not defined

In [19]:
# Quick evaluation pass with greedy decoding
if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, args.image_root, args.legend_in_prompt)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(args.output_dir, "difficult_on_easy_eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(args.output_dir, "difficult_on_easy_eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", args.output_dir)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.8903 on 1030 examples
Done.
Adapter weights are saved under: ./qwen2p5_vl_odd1out


In [18]:
import torch, os
from transformers import AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig
from peft import PeftModel

BASE = "Qwen/Qwen2.5-VL-7B-Instruct"          # or your base path
ADAPTER = "qwen2p5_vl_odd1out"  # where Trainer.save_model() wrote adapters

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

processor = AutoProcessor.from_pretrained(BASE, trust_remote_code=True)

# load base in 4-bit, then attach LoRA
model = AutoModelForVision2Seq.from_pretrained(
    BASE,
    trust_remote_code=True,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=bnb_config,
)
model = PeftModel.from_pretrained(model, ADAPTER, is_trainable=False)
model.eval()

/home/airlay88/planscape/venv/lib/python3.10/site-packages/transformers/models/auto/modeling_auto.py:2199: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2_5_VLForConditionalGeneration(
      (model): Qwen2_5_VLModel(
        (visual): Qwen2_5_VisionTransformerPretrainedModel(
          (patch_embed): Qwen2_5_VisionPatchEmbed(
            (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
          )
          (rotary_pos_emb): Qwen2_5_VisionRotaryEmbedding()
          (blocks): ModuleList(
            (0-31): 32 x Qwen2_5_VLVisionBlock(
              (norm1): Qwen2RMSNorm((1280,), eps=1e-06)
              (norm2): Qwen2RMSNorm((1280,), eps=1e-06)
              (attn): Qwen2_5_VLVisionAttention(
                (qkv): Linear4bit(in_features=1280, out_features=3840, bias=True)
                (proj): Linear4bit(in_features=1280, out_features=1280, bias=True)
              )
              (mlp): Qwen2_5_VLMLP(
                (gate_proj): lora.Linear4bit(
                  (base_layer): Linear4bit(in_features=1280, out_features=3420, bias=True)

In [19]:
df = pd.read_csv("./difficult_dataset_mid_train_test.csv")
# Basic sanitation
assert {"split", "orig_index", "outlier_id"}.issubset(df.columns), \
    "CSV must contain 'split', 'orig_index' and 'outlier_id' columns."

# # Keep only A-E rows
# df = df[df["outlier_id"].isin(LETTER_VOCAB)].copy()
LABELS = ['A','B','C','D','E']
cat = CategoricalDtype(categories=LABELS, ordered=True)

def outlier_to_letter(row):
    outlier = str(row['outlier_id'])
    opts = row['options']

    # Coerce "['...']" strings to lists if needed
    if isinstance(opts, str):
        try:
            opts = ast.literal_eval(opts)
        except Exception:
            return pd.NA
    if not isinstance(opts, (list, tuple)):
        return pd.NA

    # Normalize and only consider the first 5 (A–E)
    opts = [str(x) for x in opts][:5]
    try:
        idx = opts.index(outlier)  # 0..4 only
        return LABELS[idx]
    except ValueError:
        return pd.NA

df['outlier_id'] = df.apply(outlier_to_letter, axis=1).astype(cat)

train_df = df[df["split"].str.lower().isin(["train", "training"])]
eval_df = df[df["split"].str.lower().isin(["test", "eval", "validation", "val"])]

# if max_train_samples:
#     train_df = train_df.sample(n=min(max_train_samples, len(train_df)), random_state=args.seed)

train_df['orig_index'] += 1
eval_df['orig_index'] += 1

/tmp/ipykernel_344332/1119426447.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['orig_index'] += 1
/tmp/ipykernel_344332/1119426447.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eval_df['orig_index'] += 1


In [20]:
# Quick evaluation pass with greedy decoding
image_root = "./datasets/difficult_mix"
output_dir = "qwen2p5_vl_odd1out"

if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, image_root, False)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(output_dir, "difficult_on_difficult_mix_nolegend_eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(output_dir, "difficult_on_difficult_mix_nolegend_eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", output_dir)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.4986 on 1055 examples
Done.
Adapter weights are saved under: qwen2p5_vl_odd1out


In [21]:
# Quick evaluation pass with greedy decoding
image_root = "./datasets/difficult_reoriented"
output_dir = "qwen2p5_vl_odd1out"

if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, image_root, False)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(output_dir, "difficult_on_difficult_reoriented_nolegend_eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(output_dir, "difficult_on_difficult_reoriented_nolegend_eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", output_dir)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.5697 on 1055 examples
Done.
Adapter weights are saved under: qwen2p5_vl_odd1out


In [22]:
# Quick evaluation pass with greedy decoding
image_root = "./datasets/difficult_original"
output_dir = "qwen2p5_vl_odd1out"

if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, image_root, False)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(output_dir, "difficult_on_difficult_original_nolegend_eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(output_dir, "difficult_on_difficult_original_nolegend_eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", output_dir)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.6152 on 1055 examples
Done.
Adapter weights are saved under: qwen2p5_vl_odd1out


In [23]:
df = pd.read_csv("./easy_dataset_mid_train_test.csv")
# Basic sanitation
assert {"split", "orig_index", "outlier_id"}.issubset(df.columns), \
    "CSV must contain 'split', 'orig_index' and 'outlier_id' columns."

# # Keep only A-E rows
# df = df[df["outlier_id"].isin(LETTER_VOCAB)].copy()
LABELS = ['A','B','C','D','E']
cat = CategoricalDtype(categories=LABELS, ordered=True)

def outlier_to_letter(row):
    outlier = str(row['outlier_id'])
    opts = row['options']

    # Coerce "['...']" strings to lists if needed
    if isinstance(opts, str):
        try:
            opts = ast.literal_eval(opts)
        except Exception:
            return pd.NA
    if not isinstance(opts, (list, tuple)):
        return pd.NA

    # Normalize and only consider the first 5 (A–E)
    opts = [str(x) for x in opts][:5]
    try:
        idx = opts.index(outlier)  # 0..4 only
        return LABELS[idx]
    except ValueError:
        return pd.NA

df['outlier_id'] = df.apply(outlier_to_letter, axis=1).astype(cat)

train_df = df[df["split"].str.lower().isin(["train", "training"])]
eval_df = df[df["split"].str.lower().isin(["test", "eval", "validation", "val"])]

# if max_train_samples:
#     train_df = train_df.sample(n=min(max_train_samples, len(train_df)), random_state=args.seed)

train_df['orig_index'] += 1
eval_df['orig_index'] += 1

/tmp/ipykernel_344332/1731702007.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['orig_index'] += 1
/tmp/ipykernel_344332/1731702007.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eval_df['orig_index'] += 1


In [24]:
# Quick evaluation pass with greedy decoding
image_root = "./datasets/easy_mix"
output_dir = "qwen2p5_vl_odd1out"

if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, image_root, False)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(output_dir, "difficult_on_easy_mix_nolegend_eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(output_dir, "difficult_on_easy_mix_nolegend_eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", output_dir)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.6777 on 1030 examples
Done.
Adapter weights are saved under: qwen2p5_vl_odd1out


In [25]:
# Quick evaluation pass with greedy decoding
image_root = "./datasets/easy_original"
output_dir = "qwen2p5_vl_odd1out"

if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, image_root, False)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(output_dir, "difficult_on_easy_original_nolegend_eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(output_dir, "difficult_on_easy_original_nolegend_eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", output_dir)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.6534 on 1030 examples
Done.
Adapter weights are saved under: qwen2p5_vl_odd1out


In [26]:
# Quick evaluation pass with greedy decoding
image_root = "./datasets/easy_reoriented"
output_dir = "qwen2p5_vl_odd1out"

if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, image_root, False)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(output_dir, "difficult_on_easy_reoriented_nolegend_eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(output_dir, "difficult_on_easy_reoriented_nolegend_eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", output_dir)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.7922 on 1030 examples
Done.
Adapter weights are saved under: qwen2p5_vl_odd1out


In [6]:
BASE = "Qwen/Qwen2.5-VL-7B-Instruct"          # or your base path
ADAPTER = "qwen2p5_vl_odd1out_easy"  # where Trainer.save_model() wrote adapters

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

processor = AutoProcessor.from_pretrained(BASE, trust_remote_code=True)

# load base in 4-bit, then attach LoRA
model = AutoModelForVision2Seq.from_pretrained(
    BASE,
    trust_remote_code=True,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=bnb_config,
)
model = PeftModel.from_pretrained(model, ADAPTER, is_trainable=False)
model.eval()

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.
/home/airlay88/planscape/venv/lib/python3.10/site-packages/transformers/models/auto/modeling_auto.py:2199: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2_5_VLForConditionalGeneration(
      (model): Qwen2_5_VLModel(
        (visual): Qwen2_5_VisionTransformerPretrainedModel(
          (patch_embed): Qwen2_5_VisionPatchEmbed(
            (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
          )
          (rotary_pos_emb): Qwen2_5_VisionRotaryEmbedding()
          (blocks): ModuleList(
            (0-31): 32 x Qwen2_5_VLVisionBlock(
              (norm1): Qwen2RMSNorm((1280,), eps=1e-06)
              (norm2): Qwen2RMSNorm((1280,), eps=1e-06)
              (attn): Qwen2_5_VLVisionAttention(
                (qkv): Linear4bit(in_features=1280, out_features=3840, bias=True)
                (proj): Linear4bit(in_features=1280, out_features=1280, bias=True)
              )
              (mlp): Qwen2_5_VLMLP(
                (gate_proj): lora.Linear4bit(
                  (base_layer): Linear4bit(in_features=1280, out_features=3420, bias=True)

In [7]:
df = pd.read_csv("./difficult_dataset_mid_train_test.csv")
# Basic sanitation
assert {"split", "orig_index", "outlier_id"}.issubset(df.columns), \
    "CSV must contain 'split', 'orig_index' and 'outlier_id' columns."

# # Keep only A-E rows
# df = df[df["outlier_id"].isin(LETTER_VOCAB)].copy()
LABELS = ['A','B','C','D','E']
cat = CategoricalDtype(categories=LABELS, ordered=True)

def outlier_to_letter(row):
    outlier = str(row['outlier_id'])
    opts = row['options']

    # Coerce "['...']" strings to lists if needed
    if isinstance(opts, str):
        try:
            opts = ast.literal_eval(opts)
        except Exception:
            return pd.NA
    if not isinstance(opts, (list, tuple)):
        return pd.NA

    # Normalize and only consider the first 5 (A–E)
    opts = [str(x) for x in opts][:5]
    try:
        idx = opts.index(outlier)  # 0..4 only
        return LABELS[idx]
    except ValueError:
        return pd.NA

df['outlier_id'] = df.apply(outlier_to_letter, axis=1).astype(cat)

train_df = df[df["split"].str.lower().isin(["train", "training"])]
eval_df = df[df["split"].str.lower().isin(["test", "eval", "validation", "val"])]

# if max_train_samples:
#     train_df = train_df.sample(n=min(max_train_samples, len(train_df)), random_state=args.seed)

train_df['orig_index'] += 1
eval_df['orig_index'] += 1

/tmp/ipykernel_374897/1119426447.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['orig_index'] += 1
/tmp/ipykernel_374897/1119426447.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eval_df['orig_index'] += 1


In [10]:
# Quick evaluation pass with greedy decoding

image_root = "./datasets/difficult_mix"
output_dir = "qwen2p5_vl_odd1out_easy"

if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, image_root, False)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(output_dir, "easy_on_difficult_mix_nolegend_eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(output_dir, "easy_on_difficult_mix_nolegend_eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", output_dir)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.6550 on 1055 examples
Done.
Adapter weights are saved under: qwen2p5_vl_odd1out_easy


In [11]:
# Quick evaluation pass with greedy decoding

image_root = "./datasets/difficult_original"
output_dir = "qwen2p5_vl_odd1out_easy"

if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, image_root, False)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(output_dir, "easy_on_difficult_original_nolegend_eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(output_dir, "easy_on_difficult_original_nolegend_eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", output_dir)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.6171 on 1055 examples
Done.
Adapter weights are saved under: qwen2p5_vl_odd1out_easy


In [12]:
# Quick evaluation pass with greedy decoding

image_root = "./datasets/difficult_reoriented"
output_dir = "qwen2p5_vl_odd1out_easy"

if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, image_root, False)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(output_dir, "easy_on_difficult_reoriented_nolegend_eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(output_dir, "easy_on_difficult_reoriented_nolegend_eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", output_dir)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.8512 on 1055 examples
Done.
Adapter weights are saved under: qwen2p5_vl_odd1out_easy


In [13]:
df = pd.read_csv("./easy_dataset_mid_train_test.csv")
# Basic sanitation
assert {"split", "orig_index", "outlier_id"}.issubset(df.columns), \
    "CSV must contain 'split', 'orig_index' and 'outlier_id' columns."

# # Keep only A-E rows
# df = df[df["outlier_id"].isin(LETTER_VOCAB)].copy()
LABELS = ['A','B','C','D','E']
cat = CategoricalDtype(categories=LABELS, ordered=True)

def outlier_to_letter(row):
    outlier = str(row['outlier_id'])
    opts = row['options']

    # Coerce "['...']" strings to lists if needed
    if isinstance(opts, str):
        try:
            opts = ast.literal_eval(opts)
        except Exception:
            return pd.NA
    if not isinstance(opts, (list, tuple)):
        return pd.NA

    # Normalize and only consider the first 5 (A–E)
    opts = [str(x) for x in opts][:5]
    try:
        idx = opts.index(outlier)  # 0..4 only
        return LABELS[idx]
    except ValueError:
        return pd.NA

df['outlier_id'] = df.apply(outlier_to_letter, axis=1).astype(cat)

train_df = df[df["split"].str.lower().isin(["train", "training"])]
eval_df = df[df["split"].str.lower().isin(["test", "eval", "validation", "val"])]

# if max_train_samples:
#     train_df = train_df.sample(n=min(max_train_samples, len(train_df)), random_state=args.seed)

train_df['orig_index'] += 1
eval_df['orig_index'] += 1

/tmp/ipykernel_374897/1731702007.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['orig_index'] += 1
/tmp/ipykernel_374897/1731702007.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eval_df['orig_index'] += 1


In [14]:
# Quick evaluation pass with greedy decoding

image_root = "./datasets/easy_mix"
output_dir = "qwen2p5_vl_odd1out_easy"

if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, image_root, False)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(output_dir, "easy_on_easy_mix_nolegend_eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(output_dir, "easy_on_easy_mix_nolegend_eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", output_dir)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.8573 on 1030 examples
Done.
Adapter weights are saved under: qwen2p5_vl_odd1out_easy


In [15]:
# Quick evaluation pass with greedy decoding

image_root = "./datasets/easy_reoriented"
output_dir = "qwen2p5_vl_odd1out_easy"

if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, image_root, False)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(output_dir, "easy_on_easy_reoriented_nolegend_eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(output_dir, "easy_on_easy_reoriented_nolegend_eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", output_dir)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.9369 on 1030 examples
Done.
Adapter weights are saved under: qwen2p5_vl_odd1out_easy


In [16]:
# Quick evaluation pass with greedy decoding

image_root = "./datasets/easy_original"
output_dir = "qwen2p5_vl_odd1out_easy"

if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, image_root, False)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(output_dir, "easy_on_easy_original_nolegend_eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(output_dir, "easy_on_easy_original_nolegend_eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", output_dir)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.8408 on 1030 examples
Done.
Adapter weights are saved under: qwen2p5_vl_odd1out_easy


In [16]:
# Quick evaluation pass with greedy decoding

image_root = "./datasets/difficult_mix"
output_dir = "qwen2p5_vl_odd1out_easy"

if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, image_root, True)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(output_dir, "easy_on_difficult_mix_eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(output_dir, "easy_on_difficult_mix_eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", output_dir)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.6559 on 1055 examples
Done.
Adapter weights are saved under: qwen2p5_vl_odd1out_easy


In [19]:
# Quick evaluation pass with greedy decoding

image_root = "./datasets/easy_mix"
output_dir = "qwen2p5_vl_odd1out_easy"

if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, image_root, True)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(output_dir, "easy_on_easy_mix_eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(output_dir, "easy_on_easy_mix_eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", output_dir)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.8621 on 1030 examples
Done.
Adapter weights are saved under: qwen2p5_vl_odd1out_easy


In [20]:
# Quick evaluation pass with greedy decoding

image_root = "./datasets/easy_original"
output_dir = "qwen2p5_vl_odd1out_easy"

if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, image_root, True)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(output_dir, "easy_on_easy_original_eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(output_dir, "easy_on_easy_original_eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", output_dir)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.8437 on 1030 examples
Done.
Adapter weights are saved under: qwen2p5_vl_odd1out_easy


In [21]:
# Quick evaluation pass with greedy decoding

image_root = "./datasets/easy_reoriented"
output_dir = "qwen2p5_vl_odd1out_easy"

if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, image_root, True)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(output_dir, "easy_on_easy_reoriented_eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(output_dir, "easy_on_easy_reoriented_eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", output_dir)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.9456 on 1030 examples
Done.
Adapter weights are saved under: qwen2p5_vl_odd1out_easy


In [ ]:
image_root = "./datasets/easy"
output_dir = "qwen2p5_zeroshot"

if len(eval_df) > 0:
    eval_out = run_eval(model, processor, eval_df, image_root, True)
    acc = eval_out["accuracy"]
    print(f"[Eval] accuracy={acc:.4f} on {len(eval_out['gts'])} examples")
    with open(os.path.join(output_dir, "zeroshot_on_easy_eval_predictions.jsonl"), "w") as f:
        for idx, p, g in zip(eval_out["idxs"], eval_out["preds"], eval_out["gts"]):
            f.write(json.dumps({"orig_index": idx, "pred": p, "label": g}) + "\n")
    with open(os.path.join(output_dir, "zeroshot_on_easy_eval_metrics.json"), "w") as f:
        json.dump({"accuracy": acc, "n": len(eval_out["gts"])}, f, indent=2)

print("Done.")
print("Adapter weights are saved under:", output_dir)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

[Eval] accuracy=0.2039 on 1030 examples
Done.
Adapter weights are saved under: qwen2p5_zeroshot
